# <b><font color='teal'>Homework 20</font></b>

Объедините прогнозы, полученные с помощью моделей ARIMA, SARIMA и Prophet, чтобы повысить точность предсказаний.

In [30]:
# Импорт библиотек
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
import numpy as np

In [31]:
# Загрузите датасет в датафрейм df_Dingling
df_Dingling = pd.read_csv('../data/PRSA_Data_Dingling_20130301-20170228.csv')

display(df_Dingling)

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling
1,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling
2,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling
3,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling
4,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35059,35060,2017,2,28,19,11.0,11.0,2.0,2.0,200.0,99.0,11.7,1008.9,-13.3,0.0,NNE,1.3,Dingling
35060,35061,2017,2,28,20,13.0,13.0,2.0,2.0,200.0,101.0,10.9,1009.0,-14.0,0.0,N,2.1,Dingling
35061,35062,2017,2,28,21,9.0,14.0,2.0,2.0,200.0,102.0,9.5,1009.4,-13.0,0.0,N,1.5,Dingling
35062,35063,2017,2,28,22,10.0,12.0,2.0,2.0,200.0,97.0,7.8,1009.6,-12.6,0.0,NW,1.4,Dingling


In [32]:
# В df_Dingling создайте новый столбец datetime,
# в который соедините значения из столбцов year, month, day, hour
# Для этого используйте функцию pd.to_datetime
df_Dingling['datetime'] = pd.to_datetime(df_Dingling[['year', 'month', 'day', 'hour']])
display(df_Dingling)

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,datetime
0,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling,2013-03-01 00:00:00
1,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling,2013-03-01 01:00:00
2,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling,2013-03-01 02:00:00
3,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling,2013-03-01 03:00:00
4,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling,2013-03-01 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35059,35060,2017,2,28,19,11.0,11.0,2.0,2.0,200.0,99.0,11.7,1008.9,-13.3,0.0,NNE,1.3,Dingling,2017-02-28 19:00:00
35060,35061,2017,2,28,20,13.0,13.0,2.0,2.0,200.0,101.0,10.9,1009.0,-14.0,0.0,N,2.1,Dingling,2017-02-28 20:00:00
35061,35062,2017,2,28,21,9.0,14.0,2.0,2.0,200.0,102.0,9.5,1009.4,-13.0,0.0,N,1.5,Dingling,2017-02-28 21:00:00
35062,35063,2017,2,28,22,10.0,12.0,2.0,2.0,200.0,97.0,7.8,1009.6,-12.6,0.0,NW,1.4,Dingling,2017-02-28 22:00:00


In [33]:
# Созданный столбец сделайте индексом датафрейма (df.set_index())
df_Dingling.set_index('datetime', inplace=True)
display(df_Dingling)

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
datetime,,,,,,,,,,,,,,,,,,
2013-03-01 00:00:00,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling
2013-03-01 01:00:00,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling
2013-03-01 02:00:00,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling
2013-03-01 03:00:00,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling
2013-03-01 04:00:00,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-02-28 19:00:00,35060,2017,2,28,19,11.0,11.0,2.0,2.0,200.0,99.0,11.7,1008.9,-13.3,0.0,NNE,1.3,Dingling
2017-02-28 20:00:00,35061,2017,2,28,20,13.0,13.0,2.0,2.0,200.0,101.0,10.9,1009.0,-14.0,0.0,N,2.1,Dingling
2017-02-28 21:00:00,35062,2017,2,28,21,9.0,14.0,2.0,2.0,200.0,102.0,9.5,1009.4,-13.0,0.0,N,1.5,Dingling


In [34]:
# Поле PM2.5 выделите в отдельный объект типа Series и назовите его series_pm25
series_pm25 = df_Dingling['PM2.5']
display(series_pm25)

datetime
2013-03-01 00:00:00     4.0
2013-03-01 01:00:00     7.0
2013-03-01 02:00:00     5.0
2013-03-01 03:00:00     6.0
2013-03-01 04:00:00     5.0
                       ... 
2017-02-28 19:00:00    11.0
2017-02-28 20:00:00    13.0
2017-02-28 21:00:00     9.0
2017-02-28 22:00:00    10.0
2017-02-28 23:00:00    13.0
Name: PM2.5, Length: 35064, dtype: float64

In [35]:
# Изучите series_pm25 на наличие отсутствующих значений
series_pm25.isna().sum()

np.int64(779)

In [36]:
# Примите решение об удалении/заполнении отсутствующих значений и выполните это действие
display(df_Dingling[df_Dingling['PM2.5'].isna()])
display(df_Dingling.info())
display(df_Dingling.describe())
display(df_Dingling.duplicated().sum())



,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
datetime,,,,,,,,,,,,,,,,,,
2013-04-01 08:00:00,753,2013,4,1,8,NaN,NaN,NaN,NaN,NaN,NaN,11.0,1005.9,-3.2,0.0,NW,7.2,Dingling
2013-04-01 09:00:00,754,2013,4,1,9,NaN,NaN,NaN,NaN,NaN,NaN,10.6,1007.0,-3.5,0.0,NNW,4.5,Dingling
2013-04-01 10:00:00,755,2013,4,1,10,NaN,NaN,NaN,NaN,NaN,NaN,13.2,1007.8,-3.6,0.0,NNW,4.5,Dingling
2013-04-01 11:00:00,756,2013,4,1,11,NaN,NaN,NaN,NaN,NaN,NaN,13.8,1007.7,-4.9,0.0,NNE,4.0,Dingling
2013-04-01 12:00:00,757,2013,4,1,12,NaN,NaN,NaN,NaN,NaN,NaN,15.3,1007.9,-5.7,0.0,NE,5.0,Dingling
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-02-20 15:00:00,34864,2017,2,20,15,NaN,NaN,NaN,NaN,NaN,NaN,4.2,1022.0,-21.0,0.0,WNW,3.1,Dingling
2017-02-20 16:00:00,34865,2017,2,20,16,NaN,NaN,NaN,NaN,NaN,NaN,4.4,1022.1,-21.7,0.0,WNW,3.4,Dingling
2017-02-20 18:00:00,34867,2017,2,20,18,NaN,NaN,NaN,NaN,NaN,NaN,3.3,1022.2,-20.9,0.0,NE,1.1,Dingling


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 35064 entries, 2013-03-01 00:00:00 to 2017-02-28 23:00:00
Data columns (total 18 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   No       35064 non-null  int64  
 1   year     35064 non-null  int64  
 2   month    35064 non-null  int64  
 3   day      35064 non-null  int64  
 4   hour     35064 non-null  int64  
 5   PM2.5    34285 non-null  float64
 6   PM10     34408 non-null  float64
 7   SO2      34334 non-null  float64
 8   NO2      33830 non-null  float64
 9   CO       33052 non-null  float64
 10  O3       33850 non-null  float64
 11  TEMP     35011 non-null  float64
 12  PRES     35014 non-null  float64
 13  DEWP     35011 non-null  float64
 14  RAIN     35013 non-null  float64
 15  wd       34924 non-null  object 
 16  WSPM     35021 non-null  float64
 17  station  35064 non-null  object 
dtypes: float64(11), int64(5), object(2)
memory usage: 5.1+ MB


None

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM
count,35064.000000,35064.000000,35064.000000,35064.000000,35064.000000,34285.000000,34408.000000,34334.000000,33830.000000,33052.000000,33850.000000,35011.000000,35014.000000,35011.000000,35013.000000,35021.000000
mean,17532.500000,2014.662560,6.522930,15.729637,11.500000,65.989497,83.739723,11.749650,27.585467,904.896073,68.548371,13.686111,1007.760278,1.505495,0.060366,1.853836
std,10122.249256,1.177213,3.448752,8.800218,6.922285,72.267723,79.541685,15.519259,26.383882,903.306220,53.764424,11.365313,10.225664,13.822099,0.752899,1.309808
min,1.000000,2013.000000,1.000000,1.000000,0.000000,3.000000,2.000000,0.285600,1.026500,100.000000,0.214200,-16.600000,982.400000,-35.100000,0.000000,0.000000
25%,8766.750000,2014.000000,4.000000,8.000000,5.750000,14.000000,26.000000,2.000000,9.000000,300.000000,31.000000,3.400000,999.300000,-10.200000,0.000000,1.000000
50%,17532.500000,2015.000000,7.000000,16.000000,11.500000,41.000000,60.000000,5.000000,19.000000,600.000000,61.000000,14.700000,1007.400000,1.800000,0.000000,1.500000
75%,26298.250000,2016.000000,10.000000,23.000000,17.250000,93.000000,117.000000,15.000000,38.000000,1200.000000,90.000000,23.300000,1016.000000,14.200000,0.000000,2.300000
max,35064.000000,2017.000000,12.000000,31.000000,23.000000,881.000000,905.000000,156.000000,205.000000,10000.000000,500.000000,41.400000,1036.500000,27.200000,52.100000,10.000000


np.int64(0)

In [37]:
# Будет два варианта с dropna и fillna. Я выберу dropna, так как пропущенных значений не так много, и удаление их не повлияет на общую картину данных.
series_pm25_drop = series_pm25.dropna()
series_pm25_fill = series_pm25.interpolate(method='time').ffill().bfill()
series_pm25_fill

datetime
2013-03-01 00:00:00     4.0
2013-03-01 01:00:00     7.0
2013-03-01 02:00:00     5.0
2013-03-01 03:00:00     6.0
2013-03-01 04:00:00     5.0
                       ... 
2017-02-28 19:00:00    11.0
2017-02-28 20:00:00    13.0
2017-02-28 21:00:00     9.0
2017-02-28 22:00:00    10.0
2017-02-28 23:00:00    13.0
Name: PM2.5, Length: 35064, dtype: float64

In [59]:
# Агрегируйте данные по месяцам в переменную monthly_data
monthly_data_drop = series_pm25_drop.resample('M').sum()
display(monthly_data_drop.head(3))
monthly_data_fill = series_pm25_fill.resample('M').sum()
display(monthly_data_fill.head(3))


C:\Users\Vitaliy\AppData\Local\Temp\ipykernel_35012\239627677.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data_drop = series_pm25_drop.resample('M').sum()


datetime
2013-03-31    71205.0
2013-04-30    39031.0
2013-05-31    50410.0
Freq: ME, Name: PM2.5, dtype: float64

C:\Users\Vitaliy\AppData\Local\Temp\ipykernel_35012\239627677.py:4: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data_fill = series_pm25_fill.resample('M').sum()


datetime
2013-03-31    71205.0
2013-04-30    40104.0
2013-05-31    52551.5
Freq: ME, Name: PM2.5, dtype: float64

#### <b><font color='teal'>Модель ARIMA</font></b>

In [60]:
# Постройте модель ARIMA (назовите model_pm25_arima)
# p - компонент авторегрессии (кол-во предыдущих значений)
# d - интегральный компонент (кол-во раз, которое необходимо продифференцировать данные)
# q - компонент скользящего среднего (сколько прошлых членов ошибки используется для прогнозирования)
model_pm25_drop_arima = ARIMA(series_pm25_drop, order=(1, 1, 1)).fit()

model_pm25_fill_arima = ARIMA(series_pm25_fill, order=(1, 1, 1)).fit()

c:\Users\Vitaliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Vitaliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Vitaliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Vitaliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency info

In [61]:
# Натренируйте модель
# Результат запишите в переменную model_pm25_arima_fit
model_pm25_drop_arima_fit = ARIMA(series_pm25_drop, order=(1, 1, 1)).fit()
model_pm25_fill_arima_fit = ARIMA(series_pm25_fill, order=(1, 1, 1)).fit()   

c:\Users\Vitaliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Vitaliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Vitaliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Vitaliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency info

In [63]:
# В переменную forecast_arima_pm25 спрогнозируйте значения pm25 на ближайший год
forecast_arima_pm25_drop = model_pm25_drop_arima_fit.forecast(steps=365)
forecast_arima_pm25_fill = model_pm25_fill_arima_fit.forecast(steps=365)


c:\Users\Vitaliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\Vitaliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [64]:
# Нарисуйте график временного ряда + график прогнозов
plt.figure(figsize=(12, 6))
plt.Subplot(2, 1, 1)
plt.plot(series_pm25_drop, label='PM2.5 (dropna)', color='blue')
plt.plot(forecast_arima_pm25_drop, label='ARIMA Forecast (dropna)', color='orange')
plt.title('ARIMA Forecast for PM2.5 (dropna)')
plt.xlabel('Date')
plt.ylabel('PM2.5')
plt.legend()
plt.Subplot(2, 1, 2)
plt.plot(series_pm25_fill, label='PM2.5 (fillna)', color='green')
plt.plot(forecast_arima_pm25_fill, label='ARIMA Forecast (fillna)', color='red')
plt.title('ARIMA Forecast for PM2.5 (fillna)')
plt.xlabel('Date')
plt.ylabel('PM2.5')
plt.legend()
plt.tight_layout()
plt.show()


TypeError: subplot() takes 1 or 3 positional arguments but 2 were given

<Figure size 1200x600 with 0 Axes>

#### <b><font color='teal'>Модель SARIMA</font></b>

In [43]:
# Создайте модель SARIMA с параметрами:
# p = 1
# d = 1
# q = 1
# P = 1
# D = 1
# Q = 1
# s = 12
# Назовите переменную model_pm25_sarima


In [44]:
# Натренируйте модель на данных и запишите результат в model_pm25_sarima_fit


In [45]:
# В переменную forecast_pm25_sarima спрогнозируйте данные на 12 месяцев вперед


In [46]:
# Постройте график данных + прогнозы


#### <b><font color='teal'>Модель Prophet</font></b>

In [47]:
# Из monthly_data создайте датафрейм df_pm25 со столбцами 'ds' и 'y'


In [48]:
# Создайте экземпляр Prophet. Назовите model_prophet


In [49]:
# Обучите модель


In [50]:
# Создайте фрейм future_pm25 данных для хранения прогнозов
# с периодом 12 и с частотой в месяц (ME)


In [51]:
# Сделайте прогнозы в переменную forecast_prophet_pm25


In [52]:
# Визуализируйте прогноз и нарисуйте графики компонентов


#### <b><font color='teal'>Объединение прогнозов моделей</font></b>

In [53]:
# Подготовьте прогнозы модели Prophet к формату данных других моделей
# Создайте Series из столбца yhat (причем последних 12 значений - tail(12))
# Индексами созданного Series будет столбец ds датафрейма forecast_prophet_pm25


In [54]:
# В переменную mean_forecast вычислите среднее арифметическое всех трех прогнозов (ARIMA, SARIMA, Prophet)


In [55]:
# Постройте график данных + усредненный прогноз
